# Experiment 4 — CNN–BiLSTM–Attention Ensemble

Goal: capture local patterns with CNN, global dependencies with BiLSTM, and focus salient steps via Attention.

In [ ]:
# Core libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
plt.style.use("seaborn-v0_8")

In [ ]:
# Paths
BASE_DIR = Path('.')
TRAIN_PATH = BASE_DIR / 'train.csv'
TEST_PATH = BASE_DIR / 'test.csv'
SAMPLE_SUB_PATH = BASE_DIR / 'sample_submission .csv'

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('Train shape:', train_raw.shape)
print('Test shape:', test_raw.shape)

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    return df.drop(columns=['hour', 'month', 'dayofweek'])

def preprocess(df, is_train=True):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime')
    if 'cbwd' in df.columns:
        df['cbwd'] = df['cbwd'].fillna(method='ffill')
        df = pd.get_dummies(df, columns=['cbwd'], prefix='cbwd', drop_first=False)
    df = add_time_features(df)
    if 'No' in df.columns:
        df = df.drop(columns=['No'])
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].interpolate(limit=6, limit_direction='both')
    if is_train:
        df = df.dropna()
    else:
        df[num_cols] = df[num_cols].fillna(method='ffill').fillna(method='bfill')
    return df.reset_index(drop=True)

In [ ]:
train_df = preprocess(train_raw, is_train=True)
test_df = preprocess(test_raw, is_train=False)

split_idx = int(len(train_df) * 0.8)
train_split = train_df.iloc[:split_idx].reset_index(drop=True)
val_split = train_df.iloc[split_idx:].reset_index(drop=True)

TARGET = 'pm2.5'
LOOKBACK = 48

def scale_train_val(train_df, val_df, feature_cols):
    scaler = MinMaxScaler()
    train_scaled = train_df.copy()
    val_scaled = val_df.copy()
    train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
    return train_scaled, val_scaled, scaler

def make_sequences(df, feature_cols, target_col, lookback):
    data = df[feature_cols].values
    target = df[target_col].values
    X, y = [], []
    for i in range(lookback, len(df)):
        X.append(data[i - lookback : i])
        y.append(target[i])
    return np.array(X), np.array(y)

def inverse_target(scaler, feature_cols, y_scaled, target_col):
    target_idx = feature_cols.index(target_col)
    dummy = np.zeros((len(y_scaled), len(feature_cols)))
    dummy[:, target_idx] = y_scaled
    inv = scaler.inverse_transform(dummy)
    return inv[:, target_idx]

def evaluate_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

def plot_predictions(y_true, y_pred, title):
    plt.figure(figsize=(10, 4))
    plt.plot(y_true, label='Actual', linewidth=1)
    plt.plot(y_pred, label='Predicted', linewidth=1)
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('PM2.5')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Hybrid Model + Ensemble

In [ ]:
class TemporalAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score = layers.Dense(1)
    def call(self, inputs):
        weights = tf.nn.softmax(self.score(inputs), axis=1)
        context = tf.reduce_sum(weights * inputs, axis=1)
        return context

def build_cnn_bilstm_attention(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(filters=64, kernel_size=3, padding='causal', activation='relu')(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = TemporalAttention()(x)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1)(x)
    model = models.Model(inputs, outputs)
    return model

def recursive_forecast(model, train_scaled, test_scaled, feature_cols, target_col, lookback):
    window = train_scaled[feature_cols].iloc[-lookback:].values
    preds_scaled = []
    for i in range(len(test_scaled)):
        x = window[np.newaxis, ...]
        y_pred = model.predict(x, verbose=0).ravel()[0]
        preds_scaled.append(y_pred)
        next_row = test_scaled[feature_cols].iloc[i].copy()
        next_row[target_col] = y_pred
        window = np.vstack([window[1:], next_row.values])
    return np.array(preds_scaled)

feature_cols = train_split.columns.tolist()
train_mv_full, val_mv_full, mv_scaler_full = scale_train_val(train_split, val_split, feature_cols)
test_mv_full = test_df.copy()

for col in feature_cols:
    if col not in test_mv_full.columns:
        test_mv_full[col] = np.nan
test_mv_full = test_mv_full[feature_cols]
test_mv_full[TARGET] = train_split[TARGET].iloc[-1]
test_mv_full[feature_cols] = mv_scaler_full.transform(test_mv_full[feature_cols])

X_train_full, y_train_full = make_sequences(train_mv_full, feature_cols, TARGET, LOOKBACK)
X_val_full, y_val_full = make_sequences(val_mv_full, feature_cols, TARGET, LOOKBACK)

def train_single_model(seed):
    tf.keras.utils.set_random_seed(seed)
    model = build_cnn_bilstm_attention((X_train_full.shape[1], X_train_full.shape[2]))
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-5), loss='mse')
    es = callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    model.fit(
        X_train_full, y_train_full,
        validation_data=(X_val_full, y_val_full),
        epochs=25, batch_size=64, callbacks=[es], verbose=1
    )
    return model

ENSEMBLE_SIZE = 5
models_ens = [train_single_model(SEED + i) for i in range(ENSEMBLE_SIZE)]

val_preds = [m.predict(X_val_full, verbose=0).ravel() for m in models_ens]
val_pred_avg = np.mean(val_preds, axis=0)

y_val_true_inv = inverse_target(mv_scaler_full, feature_cols, y_val_full, TARGET)
y_val_pred_inv = inverse_target(mv_scaler_full, feature_cols, val_pred_avg, TARGET)
rmse, mae, r2 = evaluate_regression(y_val_true_inv, y_val_pred_inv)
print(f'Exp4 RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}')
plot_predictions(y_val_true_inv[:500], y_val_pred_inv[:500], 'Experiment 4 — Ensemble CNN-BiLSTM-Attention')

test_preds_scaled_all = [recursive_forecast(m, train_mv_full, test_mv_full, feature_cols, TARGET, LOOKBACK) for m in models_ens]
test_pred_scaled = np.mean(test_preds_scaled_all, axis=0)
test_pred = inverse_target(mv_scaler_full, feature_cols, test_pred_scaled, TARGET)

submission = pd.read_csv(SAMPLE_SUB_PATH)
if 'row ID' in submission.columns:
    submission['pm2.5'] = test_pred[: len(submission)]
else:
    submission = pd.DataFrame({'row ID': test_raw['datetime'].astype(str), 'pm2.5': test_pred})

sub_path = BASE_DIR / 'submission_exp4.csv'
submission.to_csv(sub_path, index=False)
print('Saved submission to:', sub_path)